In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import torch
from torch import nn
from tqdm import tqdm
from metrics import VoxEvaluation, visualizeVox
from model import *
from dataset import *

In [3]:
with open('/home/grads/wzhan24/Voxplorer/datacreate/10k_vox.pkl', 'rb') as file:  # Replace 'file_path.pkl' with the actual file path
#with open('/home/grads/wzhan24/Voxplorer/baselines/yang/microstructure_generation_3d/yang_data.pkl', 'rb') as file:
    ini_data = pickle.load(file)

In [ ]:
# Split ini_data as before
from sklearn.model_selection import train_test_split
data = np.array(ini_data)
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)
train_data = augment_rotations(train_data)
val_data = augment_rotations(val_data)
fake_train_data = fake_augment(train_data)
fake_val_data = fake_augment(val_data)
all_train_data = np.concatenate((train_data, fake_train_data), axis=0)


train_dataset = VoxelDataset(train_data)
fake_train_dataset = VoxelDataset(fake_train_data)
all_train_dataset = VoxelDataset(all_train_data)
val_dataset = VoxelDataset(val_data)
fake_val_dataset = VoxelDataset(fake_val_data)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

# Build DataLoaders
batch_size = 200
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
fake_train_loader = DataLoader(fake_train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train loader batches: {len(train_loader)}")
print(f"Fake train loader batches: {len(fake_train_loader)}")
print(f"Validation loader batches: {len(val_loader)}")

In [ ]:
# Training loop for autoencoder (encoder + decoder)
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

# Hyperparameters
latent_dim = 256
in_channels = 1  # adjust if your voxel data has different channels
num_epochs = 20 #ini 100
learning_rate = 1e-3

latent_dim=128
patch_embed_dim=128
model_dim=128
encoder = TransformerEncoder(latent_dim=latent_dim, patch_embed_dim=patch_embed_dim, model_dim=model_dim).to(device)
decoder = TransformerDecoder(latent_dim=latent_dim, patch_embed_dim=patch_embed_dim, model_dim=model_dim).to(device)

# Optimizer and loss
params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = torch.optim.Adam(params, lr=learning_rate)
criterion = nn.MSELoss()

# Training loop
repel_energy = RepelEnergy().to(device)

for epoch in tqdm(range(num_epochs)):
    encoder.train()
    decoder.train()
    total_loss = 0
    total_energy_loss = 0
    total_fake_loss = 0

    for real_batch, fake_batch in zip(train_loader, fake_train_loader):
        if real_batch.ndim == 4:
            real_batch = real_batch.unsqueeze(1)
        if fake_batch.ndim == 4:
            fake_batch = fake_batch.unsqueeze(1)
        real_batch = real_batch.to(device)
        fake_batch = fake_batch.to(device)

        optimizer.zero_grad()
        real_latent = encoder(real_batch)
        fake_latent = encoder(fake_batch)
        real_recon = decoder(real_latent)
        fake_recon = decoder(fake_latent)

        # Reconstruction loss for both real and fake
        recon_loss_real = criterion(real_recon, real_batch)
        recon_loss_fake = criterion(fake_recon, fake_batch)
        recon_loss = recon_loss_real + recon_loss_fake * 0.2

        # RepelEnergy loss
        energy_loss, inter_energy, intra_energy = repel_energy(real_latent, fake_latent)
        energy_loss_weight = 4  # Tune as needed

        #total_batch_loss = recon_loss
        total_batch_loss = recon_loss + 1e-3 * criterion(real_latent, 0.*real_latent) + \
            1e-3 * criterion(fake_latent, 0.*fake_latent) + energy_loss_weight * energy_loss

        total_batch_loss.backward()
        optimizer.step()

        total_loss += recon_loss_real.item() * real_batch.size(0)
        total_fake_loss += recon_loss_fake.item() * fake_batch.size(0)
        total_energy_loss += energy_loss.item() * real_batch.size(0)

    avg_loss = total_loss / len(train_loader.dataset)
    avg_fake_loss = total_fake_loss / len(fake_train_loader.dataset)
    avg_energy_loss = total_energy_loss / len(train_loader.dataset)
    if epoch % 2 == 0 or epoch == num_epochs - 1:
        print(f"Epoch {epoch+1}/{num_epochs} - Real Recon Loss: {avg_loss:.6f} - Fake Recon Loss: {avg_fake_loss:.6f} - Energy Loss: {avg_energy_loss:.6f}")

    if epoch % 5 == 0 or epoch == num_epochs - 1:
        torch.save(encoder.state_dict(), "0215_encoder_{}.pth".format(epoch))
        torch.save(decoder.state_dict(), "0215_decoder_{}.pth".format(epoch))

In [ ]:
train_latents = get_latents(train_loader, encoder, device)
val_latents = get_latents(val_loader, encoder, device)

train_latent_dataset = LatentDataset(train_latents)
val_latent_dataset = LatentDataset(val_latents)

print(f"Latent train dataset size: {len(train_latent_dataset)}")
print(f"Latent val dataset size: {len(val_latent_dataset)}")

Latent train dataset size: 2400
Latent val dataset size: 600


In [7]:
rp = Repellor(train_latent_dataset, n_clusters=40)
rp.cluster()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Example diffusion noise schedule (linear)
def get_beta_schedule(T, start=1e-4, end=0.02):
    return torch.linspace(start, end, T)

# Forward diffusion process: q(x_t | x_0)
def q_sample(x_0, t, betas):
    noise = torch.randn_like(x_0)
    sqrt_alphas_cumprod = torch.sqrt(torch.cumprod(1 - betas, dim=0)).to(x_0.device)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1 - torch.cumprod(1 - betas, dim=0)).to(x_0.device)
    return (
        sqrt_alphas_cumprod[t].view(-1, 1) * x_0 +
        sqrt_one_minus_alphas_cumprod[t].view(-1, 1) * noise,
        noise
    )

# Training loop for ResMLP denoising diffusion
T = 1000  # number of diffusion steps
betas = get_beta_schedule(T).to(device)
lat_dim=128
unet = ResidualMLP(input_dim=lat_dim, hidden_dim=512, output_dim=lat_dim, num_layers=16).to(device)
#unet = MLP(input_dim=latent_dim, hidden_dim=512, output_dim=latent_dim, num_layers=3).to(device)
optimizer = optim.Adam(unet.parameters(), lr=1e-3)
num_epochs = 200

train_loader_latent = DataLoader(train_latent_dataset, batch_size=200, shuffle=True)

for epoch in range(num_epochs):
    unet.train()
    total_loss = 0
    for x_0 in train_loader_latent:
        x_0 = x_0.to(device)
        batch_size = x_0.size(0)
        t = torch.randint(0, T, (batch_size,), device=device)
        x_t, noise = q_sample(x_0, t, betas)
        pred_noise = unet(x_t, t.float().unsqueeze(-1) / T)
        loss = nn.MSELoss()(pred_noise, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_size
    avg_loss = total_loss / len(train_loader_latent.dataset)
    if epoch % 10 == 0 or epoch == num_epochs - 1:
        print(f"Epoch {epoch+1}/{num_epochs} - Diffusion ResMLP Loss: {avg_loss:.6f}")

#save the diffusion model
torch.save(unet.state_dict(), "diffusion_resmlp.pth")

In [9]:
latent_dim = 128
unet = ResidualMLP(input_dim=latent_dim, hidden_dim=512, output_dim=latent_dim, num_layers=16).to(device)
#unet = MLP(input_dim=latent_dim, hidden_dim=512, output_dim=latent_dim, num_layers=3).to(device)
unet.load_state_dict(torch.load("diffusion_resmlp.pth"))
def sample_diffusion(unet, latent_dim, T, betas, device, num_samples=1000):
    unet.eval()
    x_t = torch.randn(num_samples, latent_dim).to(device)  # Start from pure noise
    for t in tqdm(reversed(range(T))):
        t_tensor = torch.full((num_samples,), t, device=device, dtype=torch.long)
        with torch.no_grad():
            pred_noise = unet(x_t, t_tensor.float().unsqueeze(-1) / T)
        alpha = 1 - betas[t]*0.01
        alpha_sqrt = torch.sqrt(alpha)
        if t > 0:
            noise = torch.randn_like(x_t)
        else:
            noise = torch.zeros_like(x_t)
        x_t = (x_t - torch.sqrt(1 - alpha) * pred_noise) / alpha_sqrt
        #x_t -= pred_noise * 2.4e-3
        x_t = x_t + torch.sqrt(betas[t]) * noise if t > 0 else x_t
        # add repelling force
        x_t_np = x_t.cpu().numpy()
        if t % 5 == 0:
            for i in range(num_samples):
                repel_force = rp.repel(x_t_np[i], threshold=100.0)
                x_t_np[i] += repel_force*0.1

    return x_t.cpu()

# Sample and decode
generated_latents = sample_diffusion(unet, latent_dim, 200, betas, device, num_samples=500)
decoder.eval()
with torch.no_grad():
    decoded_voxels = decoder(generated_latents.to(device))
    decoded_voxels = decoded_voxels.squeeze(1).cpu()

200it [00:08, 22.38it/s]


In [10]:
for i in range(decoded_voxels.shape[0]):
    decoded_voxels[i] = (decoded_voxels[i] - decoded_voxels[i].min()) / (decoded_voxels[i].max() - decoded_voxels[i].min())
    decoded_voxels[i][decoded_voxels[i] >= 0.5] = 1
    decoded_voxels[i][decoded_voxels[i] < 0.5] = 0
    #visualizeVox(decoded_voxels[i].numpy())

In [ ]:
visualizeVox(decoded_voxels[0].numpy())

In [17]:
ve = VoxEvaluation(train_dataset)

In [ ]:
sym_scores = []
per_scores = []
conn_scores = []
novelty_scores = []
#for i in tqdm(range(decoded_voxels.shape[0])):
for i in tqdm(range(200)):
    sym_scores.append(ve.symmetry_score(decoded_voxels[i]))
    per_scores.append(ve.periodicity_score(decoded_voxels[i]))
    conn_scores.append(ve.connection_score(decoded_voxels[i]))
    #novelty_scores.append(ve.novelty_score(decoded_voxels[i]))